In [14]:
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from tqdm.auto import tqdm

import sys
sys.path.insert(0, str(Path.cwd().resolve().parents[1] / '2_Propensities'))
import MF_class as MF
import OM_class as OM

np.random.seed(42)
if np.random.choice(np.arange(1000)) != 102:
    raise ValueError("Random seed is not set correctly.")

### Score function

$$
s(u,i,j,x) = [p_u; q_i]^{\top} (W + x\Delta) q_j + \alpha_u b_u + \alpha_i b_i + \alpha_j b_j + \alpha_g 
$$

where:

- $u$ : user index  
- $i$ : treatment item  
- $j$ : outcome item
- $x \in \{0,1\}$ : indicator for $u$ interacting with $i$ **and** not intercating with $j$ *before* $i$

- $p_u, q_i, q_j  \in \mathbb{R}^d$ : embedding vectors of user $u$, item $i$, item $j$

- $[p_u; q_i] \in \mathbb{R}^{2d}$ : concatenation of the user and treatment item embeddings

- $W \in \mathbb{R}^{2d \times d}$ : baseline pathway matrix mapping $(u,i)$ to item $j$  
- $\Delta \in \mathbb{R}^{2d \times d}$ : incremental pathway effect under treatment ($x=1$)

- $b_u, b_i, b_j \in \mathbb{R}$ : user, treatment item, and outcome item bias terms

- $\alpha_u, \alpha_i, \alpha_j \in \mathbb{R}$ : scaling coefficients for the bias terms

- $\alpha_g \in \mathbb{R}$ : global bias

- $s(u,i,j,x)$ : predicted logit score for the interaction.

---

### Training loss

$$
\mathcal{L}
= - \sum_n \left[
y_n \log \sigma(s_n)
+ (1 - y_n)\log(1 - \sigma(s_n))
\right]
$$

where:

- $n$ : index of a training observation  
- $y_n \in \{0,1\}$ : observed label (interaction with $j$, occurred or not)  
- $s_n$ : model logit score for observation $n$  
- $\sigma(z) = \frac{1}{1 + e^{-z}}$ : logistic sigmoid function  
- $\mathcal{L}$ : binary cross-entropy loss.

# 1. Load Dataset

In [15]:
datasets = ['ml-1m', 'steam', 'goodreads']
DATASET = datasets[0]

In [16]:
base_artifacts = Path.cwd().resolve().parents[2] / 'CausalI2I_artifacts'
data_path = base_artifacts / 'Datasets' / 'Processed' / DATASET

In [21]:
epochs_dict = {'ml-1m': 15, 'steam': 25, 'goodreads': 25}

In [22]:
om_train_data = pd.read_csv(data_path / 'om_train.csv')
om_test_data  = pd.read_csv(data_path / 'om_test.csv')

# 2. Load MF Embeddings

In [23]:
model_path = base_artifacts / 'Propensity_Models'

with open(model_path / f'MF_params_{DATASET}.pkl', 'rb') as f:
    loaded_params = pickle.load(f)

MF_model = MF.MatrixFactorizationTorch(
    n_users=loaded_params['n_users'], 
    n_items=loaded_params['n_items'], 
    n_factors=loaded_params['n_factors']
)

model_name = f'MF_model_{DATASET}'
MF_model.load(path=model_path / (model_name + '.pt'))

user_embeddings = MF_model.P.detach().numpy()[:-1]
item_embeddings = MF_model.Q.detach().numpy()[:-1]
user_bias = MF_model.b_u.detach().numpy()[:-1]
item_bias = MF_model.b_i.detach().numpy()[:-1]

Loaded model summary:
Model:                      MatrixFactorizationTorch
Number of users:            6040
Number of items:            3706
Number of factors:          40
Learning rate:              0.001
Weight decay:               1e-07
Positive weight:            1
Batch size:                 32768
Number of epochs:           25
Device:                     cuda:0
Use AMP:                    True
Timestamp:                  2026-04-11 10:50:14


# 3. Train Outcome Model

In [24]:
model = OM.OutcomeModel(
    user_embeddings = user_embeddings,
    item_embeddings = item_embeddings,
    user_bias = user_bias,
    item_bias = item_bias,
    loss = 'bce'
)

model.fit(
    df_train=om_train_data,
    df_valid=om_test_data,
    lr=2e-4,
    weight_decay=1e-4,
    epochs=epochs_dict[DATASET],
    batch_size=2**13,
)

/home/gouni/CausalI2I/4_Baselines/4.3_OutcomeModel/OM_class.py:247: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and is_cuda))


Loss: BCE
Epoch  ||- - - - - - - - Train - - - - - - - -||- - - - - - Validation - - - - - - - || Epoch's | COS θ | Time     
Number || Loss   | L-POS   | L-NEG   | MPR    || Loss   | L-POS   | L-NEG   | MPR    || Change  |       | Elapsed  
=======||========|=========|=========|========||========|=========|=========|========||=========|=======|==========
    1  || 0.4380 |  0.4546 |  0.4214 | 0.8564 || 0.3895 |  0.3983 |  0.3809 | 0.8783 ||  1.4373 | None  |    10.4s
    2  || 0.3551 |  0.3506 |  0.3595 | 0.8913 || 0.3569 |  0.3616 |  0.3523 | 0.8900 ||  0.8443 | 0.730 |    20.7s
    3  || 0.3296 |  0.3215 |  0.3377 | 0.8996 || 0.3428 |  0.3462 |  0.3394 | 0.8947 ||  0.6299 | 0.916 |    31.1s
    4  || 0.3159 |  0.3063 |  0.3255 | 0.9038 || 0.3348 |  0.3463 |  0.3237 | 0.8972 ||  0.5210 | 0.943 |    41.5s
    5  || 0.3069 |  0.2962 |  0.3175 | 0.9065 || 0.3297 |  0.3352 |  0.3243 | 0.8988 ||  0.4563 | 0.953 |    51.9s
    6  || 0.3003 |  0.2891 |  0.3114 | 0.9085 || 0.3262 |  0.3360 |

In [25]:
# with open(base_artifacts / 'Datasets' / 'Processed' / DATASET / 'item_dict.pkl', 'rb') as f:
#     item_dict = pickle.load(f)

# k = n_users
# u_list = np.arange(k)
# i = 327
# j = 135
# diffs = model.predict_outcome_differences(u_list=u_list, i=i, j=j)

# print(f"item {i} is : {item_dict[i]}")
# print(f"item {j} is : {item_dict[j]}")
# print(f"\nmean predicted outcome difference: {diffs.mean():.4f}")
# print(f"std of predicted outcome difference: {diffs.std():.4f}")

## Save Model

In [26]:
model_path = base_artifacts / 'Outcome_Models' / f'OM_{DATASET}.pt'

In [27]:
model.save(model_path, note="none")

## Load Model

In [28]:
loaded_model = OM.OutcomeModel(
    user_embeddings = user_embeddings,
    item_embeddings = item_embeddings,
    user_bias = user_bias,
    item_bias = item_bias,
    loss = 'bce'
)

loaded_model.load(model_path)

Loaded OutcomeModel summary:
Model:             OutcomeModel
Embedding dim:     40
Loss:              BCE
Learning rate:     0.0002
Weight decay:      0.0001
Batch size:        8192
Epochs:            15
Use AMP:           True
Timestamp:         2026-04-11 11:12:44
Note:              none
